In [1]:
import csv
import os
import pandas as pd
import collections
import ast
import time
import collections
import json
import sqlite3
import xxhash
import numpy as np
import pickle 
import random
import glob
from io import StringIO
from pathlib import Path
from openai import OpenAI, AsyncOpenAI
import openai
print(openai.__version__)

client = AsyncOpenAI(
    api_key=snowflake_key,
    base_url=snowflake_base_url
)




1.90.0


In [2]:
import sys
sys.path.append('./..')
from prompts.all_prompts import *
from order_by.sorting import *
from order_by.utils import *
from order_by.optimizer import *


In [3]:
import matplotlib.pyplot as plt

In [4]:
def sortedness(gold_list, sorted_data):
    kendalltau = kendalltau_distance(gold_list[:], sorted_data[:])
    return kendalltau
        
class SafeDict(dict):
    def __missing__(self, k):  # leave untouched
        return '{' + k + '}' 

In [5]:
random.seed(0)
dataname = "population_by_country_2020"

fname = f"{dataname}.csv"
file_path = f"../data/{fname}"
data = pd.read_csv(file_path)

sorted_df = data.sort_values(by=["Population (2020)", "Country"], ascending=[True, True])
print(len(sorted_df))
    
gold_list = sorted_df["Country"].tolist()
print(gold_list[:10])
shuffled_list = random.sample(gold_list, len(gold_list)) # shuffle the sorted list

200
['Saint Barthelemy', 'Nauru', 'Tuvalu', 'Anguilla', 'Palau', 'San Marino', 'Liechtenstein', 'Saint Martin', 'Monaco', 'Sint Maarten']


In [6]:
class SafeDict(dict):
    def __missing__(self, k):  # leave untouched
        return '{' + k + '}'

In [7]:
optimizer_results = {}

for sample_size in [20]:
    print(sample_size)
    print('************')
    for modelname, budget, proxy_policy in [("llama3.1-70b", 10, 'borda'), ("llama3.1-70b", 10, 'llm_judge'),
                                            ("llama3.1-405b", 20, 'borda'), ("llama3.1-405b", 20, 'llm_judge')]:
        llm_judge_prompt_template = llm_judge_prompt.format_map(SafeDict(criteria="Sort in ascending order based on the countries population in 2020"))
        
        optimizer = OrderByOptimizer(client, shuffled_list[:], 
                                    membership_inference_prompt.format_map(SafeDict(description="the 2020 population for country")),
                                    population_pointwise_prompt_template,
                                    population_external_pointwise_prompt_template,
                                    population_pairwise_comparison_prompt_template,
                                    population_external_comparison_prompt_template,
                                    budget, modelname, False, llm_judge_prompt_template, proxy_ground_truth_policy = proxy_policy, 
                                    sample_size = sample_size, ideal_oracle = gold_list, judge_model='llama3.1-405b')
    
        (sorted_data, num_api_calls, in_tokens, out_tokens), alg_name = await optimizer.physical_order_by_impl()
        kendalltau = sortedness(gold_list, sorted_data)

        if modelname not in optimizer_results:
            optimizer_results[modelname] = collections.defaultdict(list)
            optimizer_results[modelname][proxy_policy].append((sample_size, budget, kendalltau, tokens2price(modelname, in_tokens, out_tokens), alg_name))
        else:
            optimizer_results[modelname][proxy_policy].append((sample_size, budget, kendalltau, tokens2price(modelname, in_tokens, out_tokens), alg_name))
        
        print(f'{modelname} budget: {budget}')
        print(f"kendalltau score: {kendalltau}, tokens: {in_tokens+out_tokens}, price: {tokens2price(modelname, in_tokens, out_tokens)}.\n")
    print('************')

In [8]:
# cost estimation

# for sample_size in [20]:
#     print(sample_size)
#     print('************')
#     for modelname, budget, proxy_policy in [("llama3.1-70b", 10, 'borda'), ("llama3.1-70b", 10, 'llm_judge'),
#                                             ("llama3.1-405b", 20, 'borda'), ("llama3.1-405b", 20, 'llm_judge')]:
#         llm_judge_prompt_template = llm_judge_prompt.format_map(SafeDict(criteria="Sort in ascending order based on the countries population in 2020"))
        
#         optimizer = OrderByOptimizer(client, shuffled_list[:], 
#                                     membership_inference_prompt.format_map(SafeDict(description="the 2020 population for country")),
#                                     population_pointwise_prompt_template,
#                                     population_external_pointwise_prompt_template,
#                                     population_pairwise_comparison_prompt_template,
#                                     population_external_comparison_prompt_template,
#                                     budget, modelname, False, llm_judge_prompt_template, proxy_ground_truth_policy = proxy_policy, 
#                                     sample_size = sample_size, ideal_oracle = gold_list, judge_model='llama3.1-405b', run_all = True)
    
#         (sorted_data, num_api_calls, in_tokens, out_tokens), alg_name = await optimizer.physical_order_by_impl()
#         kendalltau = sortedness(gold_list, sorted_data)

#         if modelname not in optimizer_results:
#             optimizer_results[modelname] = collections.defaultdict(list)
#             optimizer_results[modelname][proxy_policy].append((sample_size, budget, kendalltau, tokens2price(modelname, in_tokens, out_tokens), alg_name))
#         else:
#             optimizer_results[modelname][proxy_policy].append((sample_size, budget, kendalltau, tokens2price(modelname, in_tokens, out_tokens), alg_name))
        
#         print(f'{modelname} budget: {budget}')
#         print(f"kendalltau score: {kendalltau}, tokens: {in_tokens+out_tokens}, price: {tokens2price(modelname, in_tokens, out_tokens)}.\n")
#     print('************')

20
************
membership count: 20
Not in training data!
membership inference cost 0.001365
remaining budget 9.998635
Final memory size for external pointwise: m = 2 (agreement = 0.50), modelname: llama3.1-70b
invoked budget 0.022375
remaining budget after initial invocation 9.97626
ext_bubble batch_size: 4
ext_bubble_4 estimated cost 0.73
ext_merge_4 estimated cost 0.03321928094887362
additional_invoked_budget 0.007738
quick_3 0.013908
quick 0.006593
ext_point 0.000805
point 0.001069
ext_merge_4 0.001866
ext_bubble_4 0.005872
quick_3 estimated cost 0.32002725179694647
quick estimated cost 0.15170690761412625
ext_point estimated cost 0.00805
point estimated cost 0.01069
ext_merge_4 estimated cost 0.037320000000000006
ext_bubble batch_size: 4
ext_bubble_4 estimated cost 0.5871999999999999
------------------
decided to use quick_3
------------------
running final decision algorithm quick_3 cost 0.340732
alg name quick_3 real cost 0.372603 alg cost est {'ext_bubble_4': 0.587199999999999

In [10]:
async def experiment(dataname, sort_by_columns, column_to_compare, pointwise_prompt_template,\
               external_pointwise_prompt_template, external_comparison_prompt_template,\
               pairwise_comparison_prompt_template, tolerance = 0.01, out_t = float):
    random.seed(0)
    
    fname = f"{dataname}.csv"
    file_path = f"../data/{fname}"
    data = pd.read_csv(file_path)

    sorted_df = data.sort_values(by=sort_by_columns, ascending=[True, True])
    print(len(sorted_df))
    
    gold_list = sorted_df[column_to_compare].tolist()
    print(gold_list[:10])
    shuffled_list = random.sample(gold_list, len(gold_list)) # shuffle the sorted list

    def add_result(results, model, alg, acc, price, token):
        results[model]['alg'].append(alg)
        results[model]['acc'].append(acc)
        results[model]['price'].append(price)
        results[model]['token'].append(token)
    results = {}
    all_results = []
    metadata = []

    memory_sizes = [4, 8]
    
    model_names = ["llama3.1-70b", 'llama3.1-405b']

    tasks = []
    for modelname in model_names:
        results[modelname] = {'alg':[], 'price':[], 'token':[], 'acc':[]}
        # 1. pointwise_sort
        tasks.append((modelname, 'point', pointwise_sort(shuffled_list[:], client, pointwise_prompt_template, modelname, out_t)))
        # 2. external_pointwise_sort
        tasks.append((modelname, 'ext_point', external_pointwise_sort(shuffled_list[:], external_values, client,
                                                                      external_pointwise_prompt_template, modelname, out_t, tolerance)))
    metadata.extend([(model, alg) for model, alg, _ in tasks])
    coroutines = [coro for _, _, coro in tasks]
    print(f"Starting {len(coroutines)} concurrent tasks...")
    res = await asyncio.gather(*coroutines)
    all_results.extend(res)

    tasks = []
    for modelname in model_names:
        for m in memory_sizes:
            tasks.append((modelname, f'ext_bubble_{m}', external_bubble_sort(shuffled_list[:], external_comparisons, m, client,
                                                                             external_comparison_prompt_template, modelname)))
    metadata.extend([(model, alg) for model, alg, _ in tasks])
    coroutines = [coro for _, _, coro in tasks]
    print(f"Starting {len(coroutines)} concurrent tasks...")
    res = await asyncio.gather(*coroutines)
    all_results.extend(res)


    tasks = []
    for modelname in model_names:
        # 3. external_bubble_sort and 4. external_merge_sort
        for m in memory_sizes:
            if m==8 and '70' in modelname:
                tasks.append((modelname, f'ext_merge_{m}', external_merge_sort(shuffled_list[:], external_comparisons, m, client,
                                                                           external_comparison_prompt_template, modelname)))
            else:
                tasks.append((modelname, f'ext_merge_{m}', external_merge_sort(shuffled_list[:], external_comparisons, m, client,
                                                                           external_comparison_prompt_template, modelname)))
    metadata.extend([(model, alg) for model, alg, _ in tasks])
    coroutines = [coro for _, _, coro in tasks]
    print(f"Starting {len(coroutines)} concurrent tasks...")
    res = await asyncio.gather(*coroutines)
    all_results.extend(res)

    tasks = []
    for modelname in model_names:
        # 5. quick_sort (vote=1)
        tasks.append((modelname, 'quick', quick_sort(shuffled_list[:], client, pairwise_comparison_prompt_template, modelname, False, 1)))

    metadata.extend([(model, alg) for model, alg, _ in tasks])
    coroutines = [coro for _, _, coro in tasks]
    print(f"Starting {len(coroutines)} concurrent tasks...")
    res = await asyncio.gather(*coroutines)
    all_results.extend(res)

    tasks = []
    for modelname in model_names:
        # 6. quick_sort (vote=3)
        tasks.append((modelname, 'quick_3', quick_sort(shuffled_list[:], client, pairwise_comparison_prompt_template, modelname, False, 3)))

    metadata.extend([(model, alg) for model, alg, _ in tasks])
    coroutines = [coro for _, _, coro in tasks]
    print(f"Starting {len(coroutines)} concurrent tasks...")
    res = await asyncio.gather(*coroutines)
    all_results.extend(res)
    
    print("All tasks completed. Processing results...")

    # Process results sequentially based on the original task order (stored in metadata)
    for i, (sorted_data, num_api_calls, in_tokens, out_tokens) in enumerate(all_results):
        modelname, alg_name = metadata[i]
        
        # Calculate Kendall-Tau
        kendalltau = sortedness(gold_list, sorted_data)
        
        # Calculate price and total tokens
        total_tokens = in_tokens + out_tokens
        price = tokens2price(modelname, in_tokens, out_tokens)
        
        # Print and store result
        print(f"[{modelname}] {alg_name}: kendalltau={kendalltau:.4f}, tokens={total_tokens}, price={price:.6f}, API Calls={num_api_calls}")
        add_result(results, modelname, alg_name, kendalltau, price, total_tokens)
    
    print("\n--- Final Results Structure ---")
    return results


In [ ]:
dataname = "population_by_country_2020"
column_to_sort = "Country"
exp_results = await experiment(
         dataname,
        ["Population (2020)", "Country"],
        "Country",
        population_pointwise_prompt_template,
        population_external_pointwise_prompt_template, 
        population_external_comparison_prompt_template,
        population_pairwise_comparison_prompt_template,
        1.0,
        float
    )

200
['Saint Barthelemy', 'Nauru', 'Tuvalu', 'Anguilla', 'Palau', 'San Marino', 'Liechtenstein', 'Saint Martin', 'Monaco', 'Sint Maarten']
Starting 4 concurrent tasks...
Final memory size for external pointwise: m = 16, modelname: llama3.1-70b
Final memory size for external pointwise: m = 16, modelname: llama3.1-405b
Starting 4 concurrent tasks...
external comparisons llama3.1-70b [ERROR] Attempt 1: 1 validation error for ExternalComparisonReasoning
  Invalid JSON: EOF while parsing a value at line 24258 column 4 [type=json_invalid, input_value='{\n  "Reasoning_steps": ...n\n    \n\n\n\n\n\n    ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/json_invalid


In [ ]:
print(exp_results)

# Figures

In [ ]:
from scipy.optimize import curve_fit

for modelname in exp_results.keys():
    alg2marker = {
        'point': 'o',
        'ext_point': '^',
        'quick': 's',
        'ext_bubble': '*',
        'ext_merge': 'D',
        'borda': 'H',
        'judge': 'x',
        'oracle': 'P', 
        'bt':'P',
    }
    
    kendalltau = exp_results[modelname]['acc']
    tokens = exp_results[modelname]['token']
    prices = exp_results[modelname]['price']
    dot_labels = exp_results[modelname]['alg']
    markers = []
    for alg in dot_labels:
        if alg[-1].isdigit():
            alg = '_'.join(alg.split('_')[:-1])
        markers.append(alg2marker[alg])

    # parse optimizer results
    opt_acc = []
    opt_prices = []
    opt_algs = []
    for policy in optimizer_results[modelname].keys():
        if policy == 'ideal':
            continue
        for arr in optimizer_results[modelname][policy]:
            if arr[0] == 20:
                _, budget, acc, price, alg_name = arr
                opt_acc.append(acc)
                opt_prices.append(price)
                opt_algs.append(policy )
    for i in range(len(opt_algs)):
        if opt_algs[i] == 'llm_judge':
            opt_algs[i] = 'judge'
        elif opt_algs[i] == 'ideal':
            opt_algs[i] = 'oracle'
            
    # Create scatter plot
    plt.figure(figsize=(8, 6))

    for x, y, label, m in zip(prices, kendalltau, dot_labels, markers):
        plt.scatter(x, y, s=100, marker=m, label=label)

    for x, y, label in zip(opt_prices, opt_acc, opt_algs):
        m = alg2marker[label]
        plt.scatter(x, y, s=100, marker=m, label=label, color='k')


    text_x, text_y = 0.6, 0.7
    ax = plt.gca()
    for x, y in zip(opt_prices, opt_acc):
        ax.annotate(
            "",
            xy=(x, y),               # data coordinates (point)
            xycoords="data",
            xytext=(text_x, text_y), # axes fraction coordinates (label position)
            textcoords=ax.transAxes,
            arrowprops=dict(
                arrowstyle="->",
                lw=1.2,
                linestyle="--",
            ),
        )
    
    ax.text(
        text_x, text_y,
        "optimizer",
        transform=ax.transAxes,   # <-- IMPORTANT
        fontsize=12,
        ha="left",
        va="center"
    )

    plt.legend(title="Labels")

    # Axis labels and title
    plt.xlabel("Price($)")
    plt.ylabel("Kendalltau")
    plt.title(f"{modelname}")    
    # Show grid and plot
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.savefig(f'{modelname}OptimizerPopulationUnlimited.png')
    plt.show()
